# 03 · MC3D dataset + pseudo coverage analysis

Characterising the MC3D PBEsol dataset and PD-A pseudo suitability. NOTE: the MC3D archive is not loaded in the current `presto` profile, so most cells here are NOT re-runnable — the stored cell outputs and data/processed/*.pkl are the record. Sections: pseudo element coverage (PD-A vs SSSP vs SR3plus), MC3D v2 element coverage & the 0.0% PD-A gap, MC3D v2 magnetism classification of all 33,142 structures, MC3D v1 magnetism analysis, and the v1-vs-v2 comparison.

*Consolidated pseudo-analysis.ipynb,mc3d-pbesol-v2-analysis.ipynb,mc3d-pbesol-v1-analysis.ipynb,mc3d-v1-v2-comparison.ipynb (2026-06-16 notebook cleanup).*

---
## A — Pseudo family element coverage (PD-A vs SSSP vs SR3plus)

In [2]:
# Load the default AiiDA profile and inspect what's registered
# (computers, codes, pseudo families)
from aiida import load_profile, orm
from aiida.orm import QueryBuilder

profile = load_profile()
print(f"Profile     : {profile.name}")
print(f"Storage     : {profile.storage_backend}")
print(f"Default user: {profile.default_user_email}")

print("\n=== Computers ===")
for c in orm.Computer.collection.all():
    print(f"  {c.label:<12}  host={c.hostname}  transport={c.transport_type}  scheduler={c.scheduler_type}")

print("\n=== Codes ===")
qb = QueryBuilder().append(orm.InstalledCode)
for code in qb.all(flat=True):
    print(f"  {code.full_label:<22}  -> {code.filepath_executable}")

print("\n=== Pseudo families ===")
qb = QueryBuilder().append(orm.Group, filters={'type_string': {'like': 'pseudo%'}})
families = sorted(qb.all(flat=True), key=lambda g: g.label)
print(f"  total: {len(families)}")
for g in families:
    print(f"  {g.label:<45}  {len(g.nodes):>3} pseudos")


Profile     : aiida_personal
Storage     : core.psql_dos
Default user: yin-junwen@outlook.com

=== Computers ===
  scarf         host=ui1.scarf.rl.ac.uk  transport=core.ssh  scheduler=core.slurm

=== Codes ===
  qe-7.2-pw@scarf         -> /work4/scd/scarf562/eb-amd/software/QuantumESPRESSO/7.2-foss-2023a/bin/pw.x

=== Pseudo families ===
  total: 18
  PseudoDojo/0.4/LDA/SR/standard/upf              70 pseudos
  PseudoDojo/0.4/LDA/SR/stringent/upf             70 pseudos
  PseudoDojo/0.4/PBE/FR/standard/upf              70 pseudos
  PseudoDojo/0.4/PBE/FR/stringent/upf             72 pseudos
  PseudoDojo/0.4/PBE/SR/standard/upf              72 pseudos
  PseudoDojo/0.4/PBE/SR/stringent/upf             72 pseudos
  PseudoDojo/0.4/PBE/SR3plus/standard/upf         14 pseudos
  PseudoDojo/0.4/PBEsol/FR/standard/upf           71 pseudos
  PseudoDojo/0.4/PBEsol/FR/stringent/upf          71 pseudos
  PseudoDojo/0.4/PBEsol/SR/standard/upf           72 pseudos
  PseudoDojo/0.4/PBEsol/SR/stringent/u

In [4]:
# Cell 2: Compare element coverage between SSSP/1.3/PBEsol/precision
# and every PseudoDojo family. Highlight diff vs Phase A's family.
from aiida.orm import Group, QueryBuilder


def family_elements(label: str) -> set[str]:
    """Return the set of element symbols covered by a pseudo family."""
    group = QueryBuilder().append(Group, filters={'label': label}).first(flat=True)
    return {pseudo.element for pseudo in group.nodes}


sssp_label = 'SSSP/1.3/PBEsol/precision'
sssp = family_elements(sssp_label)

pd_families = sorted(
    g.label for g in QueryBuilder()
        .append(Group, filters={'label': {'like': 'PseudoDojo%'}})
        .all(flat=True)
)

print(f"SSSP family : {sssp_label}  ({len(sssp)} elements)")
print()
print(f"{'PseudoDojo family':<45} {'PD':>4}  {'SSSP-only':>10}  {'PD-only':>8}  {'shared':>7}")
print("-" * 80)
for label in pd_families:
    pd = family_elements(label)
    print(f"{label:<45} {len(pd):>4}  {len(sssp - pd):>10}  {len(pd - sssp):>8}  {len(sssp & pd):>7}")

# Phase A's family — detailed element-level diff
phase_a_label = 'PseudoDojo/0.4/PBEsol/SR/standard/upf'
pd_a = family_elements(phase_a_label)

print()
print("=== Phase A family vs SSSP — element-level diff ===")
print(f"  Phase A : {phase_a_label}  ({len(pd_a)} elements)")
print(f"  SSSP    : {sssp_label}  ({len(sssp)} elements)")
print()
print(f"  in SSSP only ({len(sssp - pd_a):>2}): {sorted(sssp - pd_a)}")
print(f"  in PD-A only ({len(pd_a - sssp):>2}): {sorted(pd_a - sssp)}")
print(f"  shared       ({len(sssp & pd_a):>2}): {sorted(sssp & pd_a)}")


SSSP family : SSSP/1.3/PBEsol/precision  (103 elements)

PseudoDojo family                               PD   SSSP-only   PD-only   shared
--------------------------------------------------------------------------------
PseudoDojo/0.4/LDA/SR/standard/upf              70          33         0       70
PseudoDojo/0.4/LDA/SR/stringent/upf             70          33         0       70
PseudoDojo/0.4/PBE/FR/standard/upf              70          33         0       70
PseudoDojo/0.4/PBE/FR/stringent/upf             72          31         0       72
PseudoDojo/0.4/PBE/SR/standard/upf              72          31         0       72
PseudoDojo/0.4/PBE/SR/stringent/upf             72          31         0       72
PseudoDojo/0.4/PBE/SR3plus/standard/upf         14          89         0       14
PseudoDojo/0.4/PBEsol/FR/standard/upf           71          32         0       71
PseudoDojo/0.4/PBEsol/FR/stringent/upf          71          32         0       71
PseudoDojo/0.4/PBEsol/SR/standard/upf     

In [5]:
# Cell 3: SR3plus (lanthanides-only family) — diff against SSSP and Phase A's family
sr3plus_label = 'PseudoDojo/0.4/PBE/SR3plus/standard/upf'
pd_sr3 = family_elements(sr3plus_label)

print("=== SR3plus vs SSSP ===")
print(f"  SR3plus : {sr3plus_label}  ({len(pd_sr3)} elements)")
print(f"  SSSP    : {sssp_label}  ({len(sssp)} elements)")
print(f"  in SSSP only    ({len(sssp - pd_sr3):>3}): {sorted(sssp - pd_sr3)}")
print(f"  in SR3plus only ({len(pd_sr3 - sssp):>3}): {sorted(pd_sr3 - sssp)}")
print(f"  shared          ({len(sssp & pd_sr3):>3}): {sorted(sssp & pd_sr3)}")

print()
print("=== SR3plus vs Phase A (PD-A: PBEsol/SR/standard) — are they disjoint? ===")
print(f"  SR3plus : {sr3plus_label}  ({len(pd_sr3)} elements)")
print(f"  PD-A    : {phase_a_label}  ({len(pd_a)} elements)")
print(f"  overlap          ({len(pd_a & pd_sr3):>3}): {sorted(pd_a & pd_sr3)}")
print(f"  PD-A union SR3plus  ({len(pd_a | pd_sr3):>3} elements total)")


=== SR3plus vs SSSP ===
  SR3plus : PseudoDojo/0.4/PBE/SR3plus/standard/upf  (14 elements)
  SSSP    : SSSP/1.3/PBEsol/precision  (103 elements)
  in SSSP only    ( 89): ['Ac', 'Ag', 'Al', 'Am', 'Ar', 'As', 'At', 'Au', 'B', 'Ba', 'Be', 'Bi', 'Bk', 'Br', 'C', 'Ca', 'Cd', 'Cf', 'Cl', 'Cm', 'Co', 'Cr', 'Cs', 'Cu', 'Es', 'F', 'Fe', 'Fm', 'Fr', 'Ga', 'Ge', 'H', 'He', 'Hf', 'Hg', 'I', 'In', 'Ir', 'K', 'Kr', 'La', 'Li', 'Lr', 'Md', 'Mg', 'Mn', 'Mo', 'N', 'Na', 'Nb', 'Ne', 'Ni', 'No', 'Np', 'O', 'Os', 'P', 'Pa', 'Pb', 'Pd', 'Po', 'Pt', 'Pu', 'Ra', 'Rb', 'Re', 'Rh', 'Rn', 'Ru', 'S', 'Sb', 'Sc', 'Se', 'Si', 'Sn', 'Sr', 'Ta', 'Tc', 'Te', 'Th', 'Ti', 'Tl', 'U', 'V', 'W', 'Xe', 'Y', 'Zn', 'Zr']
  in SR3plus only (  0): []
  shared          ( 14): ['Ce', 'Dy', 'Er', 'Eu', 'Gd', 'Ho', 'Lu', 'Nd', 'Pm', 'Pr', 'Sm', 'Tb', 'Tm', 'Yb']

=== SR3plus vs Phase A (PD-A: PBEsol/SR/standard) — are they disjoint? ===
  SR3plus : PseudoDojo/0.4/PBE/SR3plus/standard/upf  (14 elements)
  PD-A    : PseudoDojo/0.4/P

In [6]:
# Cell 4 (when MC3D is imported): count MC3D structures affected by PD-A element gap
# Requires MC3D archive imported into AiiDA (separate step, not done yet)
missing_from_pd_a = sssp - pd_a
print(f"PD-A is missing {len(missing_from_pd_a)} elements:")
print(f"  {sorted(missing_from_pd_a)}")

# Once MC3D is imported as a Group, query structures containing any missing element
# (placeholder — needs MC3D archive load)


PD-A is missing 31 elements:
  ['Ac', 'Am', 'At', 'Bk', 'Ce', 'Cf', 'Cm', 'Dy', 'Er', 'Es', 'Eu', 'Fm', 'Fr', 'Gd', 'Ho', 'Lr', 'Md', 'Nd', 'No', 'Np', 'Pa', 'Pm', 'Pr', 'Pu', 'Ra', 'Sm', 'Tb', 'Th', 'Tm', 'U', 'Yb']


---
## B — MC3D PBEsol v2: element coverage + magnetism classification

In [2]:
# Each notebook needs to load the AiiDA profile before any ORM access
from aiida import load_profile
load_profile();


In [3]:
# Cell 4: Count MC3D structures affected by PD-A's element gap
# 28 effective gap elements: 13 mid-lanthanides + 15 actinides
# (At/Fr/Ra excluded since MC3D in practice has none)
from aiida.orm import Group, QueryBuilder, StructureData

MID_LANTHANIDES = {'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd',
                   'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb'}
ACTINIDES = {'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm',
             'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No', 'Lr'}
EDGE_RADIOACTIVES = {'At', 'Fr', 'Ra'}

mc3d_label = 'mc3d-pbesol-v2-structures'

qb = QueryBuilder()
qb.append(Group, filters={'label': mc3d_label}, tag='g')
qb.append(StructureData, with_group='g')

total = qb.count()
print(f"MC3D group: {mc3d_label}")
print(f"  total structures: {total}")
print(f"  scanning element sets...")

n_lanth = n_actin = n_edge = n_any = 0
for (struct,) in qb.iterall():
    elements = struct.get_symbols_set()       # set of element symbols, not kind names
    hit_lanth = bool(elements & MID_LANTHANIDES)
    hit_actin = bool(elements & ACTINIDES)
    hit_edge = bool(elements & EDGE_RADIOACTIVES)
    n_lanth += hit_lanth
    n_actin += hit_actin
    n_edge += hit_edge
    n_any += (hit_lanth or hit_actin or hit_edge)

print()
print("Structures affected by PD-A element gap:")
print(f"  contains any mid-lanthanide  (13 elements): {n_lanth:>5}  ({100*n_lanth/total:5.1f}%)")
print(f"  contains any actinide        (15 elements): {n_actin:>5}  ({100*n_actin/total:5.1f}%)")
print(f"  contains any edge radioactive ( 3 elements): {n_edge:>5}  ({100*n_edge/total:5.1f}%)")
print(f"  -------------------------------------------- ")
print(f"  contains any of the 31 gap elements        : {n_any:>5}  ({100*n_any/total:5.1f}%)")


MC3D group: mc3d-pbesol-v2-structures
  total structures: 33142
  scanning element sets...

Structures affected by PD-A element gap:
  contains any mid-lanthanide  (13 elements):     0  (  0.0%)
  contains any actinide        (15 elements):     0  (  0.0%)
  contains any edge radioactive ( 3 elements):     0  (  0.0%)
  -------------------------------------------- 
  contains any of the 31 gap elements        :     0  (  0.0%)


In [4]:
# Cell 3: total unique elements across all MC3D PBEsol v2 structures
from aiida.orm import Group, QueryBuilder, StructureData

# Sanity sample: first 3 structures, print their element sets explicitly
qb = QueryBuilder()
qb.append(Group, filters={'label': 'mc3d-pbesol-v2-structures'}, tag='g')
qb.append(StructureData, with_group='g')
sample = qb.limit(3).all(flat=True)
print("Sanity sample (first 3 structures):")
for s in sample:
    print(f"  uuid={s.uuid[:8]}  formula={s.get_formula()}  symbols={sorted(s.get_symbols_set())}")

# Full scan: union of element symbols across all structures
qb = QueryBuilder()
qb.append(Group, filters={'label': 'mc3d-pbesol-v2-structures'}, tag='g')
qb.append(StructureData, with_group='g')
all_elements = set()
n = 0
for (struct,) in qb.iterall():
    all_elements |= struct.get_symbols_set()
    n += 1

print(f"\nScanned {n} structures")
print(f"Total unique elements in MC3D PBEsol v2: {len(all_elements)}")
print(f"Sorted: {sorted(all_elements)}")

# Cross-check vs PD-A (the 72 elements PD-A supplies, from earlier shared list)
PD_A_72 = {
    'Ag','Al','Ar','As','Au','B','Ba','Be','Bi','Br','C','Ca','Cd','Cl','Co','Cr','Cs','Cu',
    'F','Fe','Ga','Ge','H','He','Hf','Hg','I','In','Ir','K','Kr','La','Li','Lu','Mg','Mn',
    'Mo','N','Na','Nb','Ne','Ni','O','Os','P','Pb','Pd','Po','Pt','Rb','Re','Rh','Rn','Ru',
    'S','Sb','Sc','Se','Si','Sn','Sr','Ta','Tc','Te','Ti','Tl','V','W','Xe','Y','Zn','Zr',
}
print(f"\nMC3D − PD-A: {sorted(all_elements - PD_A_72)} (should be empty if MC3D pre-filtered to PD-A)")
print(f"PD-A − MC3D: {sorted(PD_A_72 - all_elements)} (PD-A elements not present in any MC3D structure)")


Sanity sample (first 3 structures):
  uuid=0290c725  formula=BaCd  symbols=['Ba', 'Cd']
  uuid=18de2078  formula=AlB2Cr2  symbols=['Al', 'B', 'Cr']
  uuid=1979bdca  formula=CsF3Fe  symbols=['Cs', 'F', 'Fe']

Scanned 33142 structures
Total unique elements in MC3D PBEsol v2: 70
Sorted: ['Ag', 'Al', 'Ar', 'As', 'Au', 'B', 'Ba', 'Be', 'Bi', 'Br', 'C', 'Ca', 'Cd', 'Cl', 'Co', 'Cr', 'Cs', 'Cu', 'F', 'Fe', 'Ga', 'Ge', 'H', 'He', 'Hf', 'Hg', 'I', 'In', 'Ir', 'K', 'Kr', 'Li', 'Mg', 'Mn', 'Mo', 'N', 'Na', 'Nb', 'Ne', 'Ni', 'O', 'Os', 'P', 'Pb', 'Pd', 'Po', 'Pt', 'Rb', 'Re', 'Rh', 'Rn', 'Ru', 'S', 'Sb', 'Sc', 'Se', 'Si', 'Sn', 'Sr', 'Ta', 'Tc', 'Te', 'Ti', 'Tl', 'V', 'W', 'Xe', 'Y', 'Zn', 'Zr']

MC3D − PD-A: [] (should be empty if MC3D pre-filtered to PD-A)
PD-A − MC3D: ['La', 'Lu'] (PD-A elements not present in any MC3D structure)


In [4]:
# Inspect the two PwBaseWorkChain stages for mc3d-33524 (Na2O) side-by-side.

import numpy as np
from aiida.orm import load_node, QueryBuilder, Group, WorkChainNode

PWBASE_UUIDS = [
    '09abff7e-b6b9-46a6-a748-fc9cfa1df626',  # stage 1
    'fa5cb995-2680-439b-a005-df3e3f2b8147',  # stage 2
]

for i, uuid in enumerate(PWBASE_UUIDS, start=1):
    print(f'\n========== PwBase stage {i}  uuid={uuid} ==========')
    try:
        wc = load_node(uuid)
    except Exception as e:
        print(f'  NOT IN DB: {e}')
        continue

    print(f'  pk={wc.pk}  process_label={wc.process_label}  '
          f'state={wc.process_state.value if wc.process_state else "?"}  '
          f'exit={wc.exit_status}')

    qg = QueryBuilder()
    qg.append(WorkChainNode, filters={'id': wc.pk}, tag='wc')
    qg.append(Group, with_node='wc', project=['label'])
    print(f'  groups: {[r[0] for r in qg.all()]}')

    in_struct = wc.inputs.pw.structure
    mid = in_struct.base.extras.all.get('mc3d_id', '<none>')
    print(f'  INPUT  StructureData pk={in_struct.pk}  '
          f'formula={in_struct.get_formula()}  mc3d_id={mid}')

    if 'output_structure' in wc.outputs:
        out_struct = wc.outputs.output_structure
        omid = out_struct.base.extras.all.get('mc3d_id', '<none>')
        print(f'  OUTPUT StructureData pk={out_struct.pk}  '
              f'formula={out_struct.get_formula()}  mc3d_id={omid}')
    else:
        print(f'  OUTPUT StructureData: <none>')

    p = wc.outputs.output_parameters.get_dict()
    keys_of_interest = [
        'lsda', 'do_magnetization', 'number_of_spin_components',
        'starting_magnetization', 'constraint_mag',
        'total_magnetization', 'absolute_magnetization',
        'energy', 'fermi_energy', 'number_of_atoms',
        'total_number_of_scf_iterations', 'wall_time_seconds',
    ]
    print(f'  output_parameters (selected; total {len(p)} keys):')
    for k in keys_of_interest:
        v = p.get(k, '<missing>')
        print(f'    {k:35s} = {v}')

    if 'output_trajectory' in wc.outputs:
        traj = wc.outputs.output_trajectory
        names = traj.get_arraynames()
        print(f'  output_trajectory arrays: {names}')

        if 'energy' in names:
            e = traj.get_array('energy')
            print(f'    energy   shape={e.shape}  values={np.round(e, 5).tolist()}')
        if 'atomic_species_name' in names:
            sp = traj.get_array('atomic_species_name')
            print(f'    species  {list(sp)}')
        if 'atomic_magnetic_moments' in names:
            m = traj.get_array('atomic_magnetic_moments')
            print(f'    atomic_magnetic_moments shape={m.shape}')
            print(f'      first SCF row : {np.round(m[0], 4).tolist()}')
            print(f'      converged row : {np.round(m[-1], 4).tolist()}')
            converged = m[-1]
            print(f'      sum (signed)        = {converged.sum():+.4f}')
            print(f'      sum |.| (abs total) = {np.abs(converged).sum():.4f}')
            print(f'      max |atomic|        = {np.abs(converged).max():.4f}')
        if 'cells' in names:
            c = traj.get_array('cells')
            print(f'    cells    shape={c.shape}  '
                  f'last_cell_diag={np.round(np.diag(c[-1]), 4).tolist()}')
    else:
        print(f'  output_trajectory: <none>')



========== PwBase stage 1  uuid=09abff7e-b6b9-46a6-a748-fc9cfa1df626 ==========
  pk=36108  process_label=PwBaseWorkChain  state=finished  exit=0
  groups: ['mc3d-pbesol-v2']
  INPUT  StructureData pk=68070  formula=Na2O  mc3d_id=<none>
  OUTPUT StructureData pk=34417  formula=Na2O  mc3d_id=<none>
  output_parameters (selected; total 74 keys):
    lsda                                = True
    do_magnetization                    = False
    number_of_spin_components           = 2
    starting_magnetization              = [-5.8055578056807e-13, 3.9708852690268e-10]
    constraint_mag                      = 0
    total_magnetization                 = <missing>
    absolute_magnetization              = <missing>
    energy                              = -3151.2382914745
    fermi_energy                        = 4.4914978932344
    number_of_atoms                     = 3
    total_number_of_scf_iterations      = 41
    wall_time_seconds                   = 57.1748919487
  output_trajector

In [5]:
# Confirm QE itself emitted magnetization to QEXSD; the old parser dropped it.
from aiida.orm import load_node
import xml.etree.ElementTree as ET

wc = load_node('fa5cb995-2680-439b-a005-df3e3f2b8147')   # mc3d-33524 stage 2
ret = wc.outputs.retrieved
print(f'retrieved files: {ret.list_object_names()}')

with ret.open('data-file-schema.xml', 'r') as f:
    xml_text = f.read()

# Strip namespace to make XPath simple
xml_clean = xml_text.replace('xmlns="http://www.quantum-espresso.org/ns/qes/qes-1.0"', '')
root = ET.fromstring(xml_clean)

# QEXSD path: output/magnetization/{total,absolute,...}
mag = root.find('.//magnetization')
if mag is None:
    print('NO <magnetization> block in XML — QE itself did not emit')
else:
    print('FOUND <magnetization> block:')
    for child in mag:
        print(f'  <{child.tag}> = {child.text}')

# Also: output/band_structure has lsda/nbnd_up/nbnd_dw/two_fermi_energies
bs = root.find('.//band_structure')
if bs is not None:
    for tag in ('lsda', 'nbnd_up', 'nbnd_dw', 'noncolin', 'spinorbit'):
        e = bs.find(tag)
        if e is not None:
            print(f'  band_structure/{tag} = {e.text}')


retrieved files: ['_scheduler-stderr.txt', '_scheduler-stdout.txt', 'aiida.out', 'data-file-schema.xml']
FOUND <magnetization> block:
  <lsda> = true
  <noncolin> = false
  <spinorbit> = false
  <total> = 1.057798251575257E-15
  <absolute> = 4.423923196013534E-08
  band_structure/lsda = true
  band_structure/nbnd_up = 16
  band_structure/nbnd_dw = 16
  band_structure/noncolin = false
  band_structure/spinorbit = false


In [8]:
# Classify all 33,142 MC3D PBEsol-v2 structures by per-atom magnetic moments
# from each canonical relax WC's output_trajectory. Builds a master DataFrame
# then prints category counts.

import time
import numpy as np
import pandas as pd
from collections import Counter
from aiida.orm import (
    Group, QueryBuilder, StructureData, TrajectoryData, WorkChainNode,
)

# --- thresholds (PLAN §4-ish) ---
NM_MAX_ATOMIC      = 0.05  # no atom carries any moment
AFM_TOTAL_MAX      = 0.05  # signed total cancels
SIGNIF_ATOMIC      = 0.10  # atom is "magnetic"
FM_STRONG_TOTAL    = 0.50  # large net moment

SOURCE_GROUPS = [
    ('icsd', 'workchain/icsd/final-pw'),
    ('mpds', 'workchain/mpds/final-pw'),
    ('cod',  'workchain/cod/relax'),
]

# --- pull (mc3d_id, structure metadata, trajectory) in one QueryBuilder per source ---
rows = []
t0 = time.time()
for source, gpath in SOURCE_GROUPS:
    qb = QueryBuilder()
    qb.append(Group, filters={'label': gpath}, tag='g')
    qb.append(WorkChainNode, with_group='g', tag='wc')
    qb.append(
        StructureData,
        with_incoming='wc',
        edge_filters={'label': 'output_structure'},
        filters={'extras.mc3d_id': {'!==': None}},
        project=['extras.mc3d_id', 'extras.formula_hill_compact',
                 'extras.spacegroup_number', 'extras.number_of_sites',
                 'extras.chemical_system'],
        tag='s',
    )
    qb.append(
        TrajectoryData,
        with_incoming='wc',
        edge_filters={'label': 'output_trajectory'},
        project=['*'],
        tag='traj',
    )

    n_before = len(rows)
    for mc3d_id, formula, spg, n_sites, chem, traj in qb.iterall():
        names = traj.get_arraynames()
        if 'atomic_magnetic_moments' not in names:
            continue
        m = traj.get_array('atomic_magnetic_moments')[-1]
        species = (list(traj.get_array('atomic_species_name'))
                   if 'atomic_species_name' in names else None)

        m_abs = np.abs(m)
        # per-atom magnetic species: those carrying significant moment
        mag_atoms = ([sp for sp, mm in zip(species, m_abs) if mm > SIGNIF_ATOMIC]
                     if species is not None else [])

        rows.append({
            'mc3d_id':         mc3d_id,
            'formula':         formula,
            'spacegroup':      spg,
            'n_sites':         n_sites,
            'chemical_system': chem,
            'source':          source,
            'species':         species,
            'atomic_moments':  m.tolist(),
            'total_mag':       float(m.sum()),
            'abs_mag':         float(m_abs.sum()),
            'max_abs_atomic':  float(m_abs.max()),
            'magnetic_species': sorted(set(mag_atoms)),
        })
    print(f'  {gpath}: +{len(rows) - n_before} rows  (cumulative {len(rows)})')

print(f'\nTotal rows: {len(rows)}  in {time.time()-t0:.1f}s')

# --- categorize ---
def categorize(r):
    tot = abs(r['total_mag'])
    mx  = r['max_abs_atomic']
    if mx < NM_MAX_ATOMIC:
        return 'NM'                      # nonmagnetic
    if tot < AFM_TOTAL_MAX and mx > SIGNIF_ATOMIC:
        return 'AFM_like'                # local moments cancel
    if tot > FM_STRONG_TOTAL:
        return 'FM_strong'               # large net moment
    if mx > SIGNIF_ATOMIC:
        return 'FiM_or_partial'          # nonzero net but not strong
    return 'borderline'                  # small moments, no clear class

df = pd.DataFrame(rows)
df['category'] = df.apply(categorize, axis=1)

# --- summary ---
print('\n=== Category counts ===')
print(df['category'].value_counts().to_string())
print(f'\nMagnetic (any) :  {(df["category"] != "NM").sum():>6}  '
      f'({100 * (df["category"] != "NM").mean():.1f}%)')
print(f'Nonmagnetic    :  {(df["category"] == "NM").sum():>6}  '
      f'({100 * (df["category"] == "NM").mean():.1f}%)')

print('\n=== Top magnetic species (in magnetic structures) ===')
mag_df = df[df['category'] != 'NM']
species_counter = Counter()
for sp_list in mag_df['magnetic_species']:
    species_counter.update(sp_list)
for sp, c in species_counter.most_common(20):
    print(f'  {sp:>4}  {c:>5}  ({100 * c / len(mag_df):.1f}% of magnetic)')

print('\n=== |total_mag| distribution percentiles ===')
print(df['total_mag'].abs().describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

print('\n=== Sample of FM_strong (top 10 by |total_mag|) ===')
print(df[df['category'] == 'FM_strong']
      .sort_values('total_mag', key=abs, ascending=False)
      [['mc3d_id', 'formula', 'spacegroup', 'total_mag', 'abs_mag', 'magnetic_species']]
      .head(10).to_string(index=False))

print('\n=== Sample of AFM_like (top 10 by max_abs_atomic) ===')
print(df[df['category'] == 'AFM_like']
      .sort_values('max_abs_atomic', ascending=False)
      [['mc3d_id', 'formula', 'spacegroup', 'total_mag', 'max_abs_atomic', 'magnetic_species']]
      .head(10).to_string(index=False))
df.to_pickle('data/processed/mc3d-pbesol-v2-magnetism.pkl')


  workchain/icsd/final-pw: +15980 rows  (cumulative 15980)
  workchain/mpds/final-pw: +5382 rows  (cumulative 21362)
  workchain/cod/relax: +11780 rows  (cumulative 33142)

Total rows: 33142  in 206.1s

=== Category counts ===
category
NM                27435
FM_strong          5521
FiM_or_partial      159
borderline           27

Magnetic (any) :    5707  (17.2%)
Nonmagnetic    :   27435  (82.8%)

=== Top magnetic species (in magnetic structures) ===
    Fe   1572  (27.5% of magnetic)
    Mn   1168  (20.5% of magnetic)
     O    994  (17.4% of magnetic)
    Co    610  (10.7% of magnetic)
    Cu    572  (10.0% of magnetic)
    Cr    467  (8.2% of magnetic)
     V    462  (8.1% of magnetic)
    Ni    261  (4.6% of magnetic)
     F    245  (4.3% of magnetic)
    Ru    129  (2.3% of magnetic)
    Ti    106  (1.9% of magnetic)
    Mo    101  (1.8% of magnetic)
    Cl     82  (1.4% of magnetic)
    Ir     72  (1.3% of magnetic)
     N     67  (1.2% of magnetic)
    Re     63  (1.1% of magne

In [9]:
# Augment v2 pkl with source_db_id without redoing the 206s backfill.
# StructureData extras hold source as a nested dict: {'database': 'mpds', 'id': 'S1030151', ...}
# So source_db_id = f'{database}-{id}'.

import pandas as pd
from aiida.orm import Group, QueryBuilder, StructureData

df_v2 = pd.read_pickle('data/processed/mc3d-pbesol-v2-magnetism.pkl')

# Pull (mc3d_id -> source.database, source.id) once
qb = QueryBuilder()
qb.append(Group, filters={'label': 'mc3d-pbesol-v2-structures'}, tag='g')
qb.append(StructureData, with_group='g',
          filters={'extras.mc3d_id': {'!==': None}},
          project=['extras.mc3d_id',
                   'extras.source.database',
                   'extras.source.id'])
mp = {mid: f'{db}-{sid}' for mid, db, sid in qb.iterall() if db and sid}
print(f'mc3d_id -> source_db_id map: {len(mp)} entries')

df_v2['source_db_id'] = df_v2['mc3d_id'].map(mp)
print(f'rows with source_db_id set: {df_v2["source_db_id"].notna().sum()} / {len(df_v2)}')
print(df_v2[['mc3d_id', 'formula', 'source', 'source_db_id']].head(5).to_string())

# Save back
df_v2.to_pickle('data/processed/mc3d-pbesol-v2-magnetism.pkl')
print('\nUpdated pkl saved.')


mc3d_id -> source_db_id map: 33142 entries
rows with source_db_id set: 33134 / 33142
      mc3d_id formula source source_db_id
0  mc3d-19919     IrN   icsd  icsd-183155
1  mc3d-48310  CsF3Fe   icsd   icsd-49583
2  mc3d-38841     MnN   icsd  icsd-184928
3  mc3d-19663    SiSn   icsd  icsd-184676
4   mc3d-7912      Na   icsd  icsd-671302

Updated pkl saved.


In [10]:
# Rename v2 pkl column 'source' -> 'source_db' (clearer; matches v1 schema convention)
import pandas as pd

path = 'data/processed/mc3d-pbesol-v2-magnetism.pkl'
df_v2 = pd.read_pickle(path)
df_v2 = df_v2.rename(columns={'source': 'source_db'})
print(f'columns now: {list(df_v2.columns)}')
df_v2.to_pickle(path)
print(f'saved to {path}')


columns now: ['mc3d_id', 'formula', 'spacegroup', 'n_sites', 'chemical_system', 'source_db', 'species', 'atomic_moments', 'total_mag', 'abs_mag', 'max_abs_atomic', 'magnetic_species', 'category', 'source_db_id']
saved to data/processed/mc3d-pbesol-v2-magnetism.pkl


---
## C — MC3D PBEsol v1: magnetism analysis

In [2]:
from aiida import load_profile
load_profile();


In [4]:
from aiida.orm import Group, QueryBuilder, Node
from collections import Counter

GROUP_LABEL = 'mc3d-pbesol-v1'

g = Group.collection.get(label=GROUP_LABEL)
print(f'group pk={g.pk}  label={g.label}  type_string={g.type_string}')
print(f'description: {g.description!r}')
print(f'total nodes in group: {g.count()}')

# 统计 group 里每种 node 类型有多少个
qb = QueryBuilder()
qb.append(Group, filters={'label': GROUP_LABEL}, tag='g')
qb.append(Node, with_group='g', project=['node_type'])

types = Counter(row[0] for row in qb.iterall())
print('\nnode_type 分布:')
for t, n in types.most_common():
    print(f'  {n:>7}  {t}')


group pk=22  label=mc3d-pbesol-v1  type_string=core.import
description: 'Group generated by archive import'
total nodes in group: 684786

node_type 分布:
   186066  data.core.dict.Dict.
    71486  data.core.bool.Bool.
    70398  data.core.structure.StructureData.
    36200  data.core.folder.FolderData.
    36200  data.core.array.trajectory.TrajectoryData.
    36200  data.core.remote.RemoteData.
    36200  process.calculation.calcjob.CalcJobNode.
    36188  data.core.array.bands.BandsData.
    35792  process.workflow.workchain.WorkChainNode.
    35044  process.calculation.calcfunction.CalcFunctionNode.
    35044  data.core.array.kpoints.KpointsData.
    34946  data.core.float.Float.
    34946  data.core.int.Int.
       70  data.pseudo.upf.UpfData.
        6  data.core.code.Code.


In [6]:
from aiida.orm import Group, QueryBuilder, WorkChainNode
from collections import Counter

GROUP_LABEL = 'mc3d-pbesol-v1'

qb = QueryBuilder()
qb.append(Group, filters={'label': GROUP_LABEL}, tag='g')
qb.append(
    WorkChainNode,
    with_group='g',
    project=[
        'attributes.process_label',
        'attributes.process_state',
        'attributes.exit_status',
    ],
)

labels = Counter()
states = Counter()
exits  = Counter()
for plabel, pstate, exit_status in qb.iterall():
    labels[plabel] += 1
    states[pstate] += 1
    exits[(plabel, exit_status)] += 1

print('process_label 分布:')
for k, v in labels.most_common():
    print(f'  {v:>7}  {k}')

print('\nprocess_state 分布:')
for k, v in states.most_common():
    print(f'  {v:>7}  {k}')

print('\n(process_label, exit_status) 分布 (top 15):')
for (pl, ec), v in exits.most_common(15):
    print(f'  {v:>7}  {pl:30s}  exit={ec}')


process_label 分布:
    35044  PwBaseWorkChain
      748  PwRelaxWorkChain

process_state 分布:
    35792  finished

(process_label, exit_status) 分布 (top 15):
    32960  PwBaseWorkChain                 exit=0
     2084  PwBaseWorkChain                 exit=501
      748  PwRelaxWorkChain                exit=0


In [8]:
import numpy as np
from aiida.orm import Group, QueryBuilder, StructureData, TrajectoryData, WorkChainNode

GROUP_LABEL = 'mc3d-pbesol-v1'

# 抽一个 PwBase / exit=0 的 WC（取 PK 最小的稳定可复现）
qb = QueryBuilder()
qb.append(Group, filters={'label': GROUP_LABEL}, tag='g')
qb.append(
    WorkChainNode,
    with_group='g',
    filters={
        'attributes.process_label': 'PwBaseWorkChain',
        'attributes.exit_status':   0,
    },
    project=['*'],
    tag='wc',
)
qb.order_by({'wc': {'id': 'asc'}})
wc = qb.first(flat=True)

# --- 节点本身 ---
print(f'━━━ WorkChainNode ━━━')
print(f'  pk             = {wc.pk}')
print(f'  uuid           = {wc.uuid}')
print(f'  process_label  = {wc.process_label}')
print(f'  process_state  = {wc.process_state.value if wc.process_state else "?"}')
print(f'  exit_status    = {wc.exit_status}')
print(f'  ctime          = {wc.ctime}')

# --- 它属于哪些 group（除 v1 外）---
qg = QueryBuilder()
qg.append(WorkChainNode, filters={'id': wc.pk}, tag='n')
qg.append(Group, with_node='n', project=['label'])
print(f'  groups         = {[r[0] for r in qg.all()]}')

# --- 输入 / 输出 structure ---
print(f'\n━━━ INPUT structure ━━━')
in_s = wc.inputs.pw.structure
print(f'  pk={in_s.pk}  formula={in_s.get_formula()}  n_sites={len(in_s.sites)}')
print(f'  symbols_set    = {sorted(in_s.get_symbols_set())}')
print(f'  cell_volume    = {in_s.get_cell_volume():.3f} A^3')
print(f'  extras         = {dict(in_s.base.extras.all)}')

if 'output_structure' in wc.outputs:
    out_s = wc.outputs.output_structure
    print(f'\n━━━ OUTPUT structure ━━━')
    print(f'  pk={out_s.pk}  formula={out_s.get_formula()}  n_sites={len(out_s.sites)}')
    print(f'  cell_volume    = {out_s.get_cell_volume():.3f} A^3')
    print(f'  extras         = {dict(out_s.base.extras.all)}')
else:
    print('\n━━━ OUTPUT structure: <none>')

# --- 全部 output 链接（看看 WC 总共抛出了哪些东西）---
print(f'\n━━━ outputs.all link labels ━━━')
for label in wc.outputs:
    n = getattr(wc.outputs, label)
    print(f'  {label:25s}  {type(n).__name__}  pk={n.pk}')

# --- output_parameters: 全键 dump（aiida-scf 风格）---
p = wc.outputs.output_parameters.get_dict()
print(f'\n━━━ output_parameters: {len(p)} keys ━━━')
for k in sorted(p.keys()):
    v = p[k]
    if isinstance(v, (str, int, float, bool)) or v is None:
        print(f'  {k:40s} = {v}')
    elif isinstance(v, dict):
        print(f'  {k:40s} = <dict, keys={list(v.keys())[:5]}{"..." if len(v) > 5 else ""}>')
    elif isinstance(v, list):
        preview = v[:3] if len(v) <= 6 else f'len={len(v)}'
        print(f'  {k:40s} = <list> {preview}')
    else:
        print(f'  {k:40s} = <{type(v).__name__}>')

# --- output_trajectory: 全数组 ---
if 'output_trajectory' in wc.outputs:
    traj = wc.outputs.output_trajectory
    names = traj.get_arraynames()
    print(f'\n━━━ output_trajectory: {len(names)} arrays ━━━')
    for nm in names:
        a = traj.get_array(nm)
        print(f'  {nm:30s} shape={str(a.shape):20s} dtype={a.dtype}')

# --- 磁性收口 ---
print(f'\n━━━ MAGNETISM SUMMARY ━━━')
print(f'  lsda                       = {p.get("lsda")}')
print(f'  number_of_spin_components  = {p.get("number_of_spin_components")}')
print(f'  starting_magnetization     = {p.get("starting_magnetization")}')
print(f'  total_magnetization        = {p.get("total_magnetization")}')
print(f'  absolute_magnetization     = {p.get("absolute_magnetization")}')

if 'output_trajectory' in wc.outputs and 'atomic_magnetic_moments' in traj.get_arraynames():
    m = traj.get_array('atomic_magnetic_moments')[-1]   # 收敛那一行
    sp = list(traj.get_array('atomic_species_name'))
    print(f'  atomic_magnetic_moments (converged):')
    for s, mv in zip(sp, m):
        print(f'    {s:>4}  {mv:+.4f}')
    print(f'  sum (signed)               = {m.sum():+.4f}')
    print(f'  sum |.|                    = {np.abs(m).sum():.4f}')
    print(f'  max |atomic|               = {np.abs(m).max():.4f}')
else:
    print(f'  atomic_magnetic_moments    = <not in trajectory>')


━━━ WorkChainNode ━━━
  pk             = 805
  uuid           = 0046c2e2-7f02-478a-8d1f-6c75081c8cc8
  process_label  = PwBaseWorkChain
  process_state  = finished
  exit_status    = 0
  ctime          = 2021-09-23 17:40:35.765619+01:00
  groups         = ['20250828-131437', 'mc3d-pbesol-v1']

━━━ INPUT structure ━━━
  pk=1759  formula=Au6Zr2  n_sites=8
  symbols_set    = ['Au', 'Zr']
  cell_volume    = 141.262 A^3
  extras         = {}

━━━ OUTPUT structure ━━━
  pk=40727  formula=Au6Zr2  n_sites=8
  cell_volume    = 141.262 A^3
  extras         = {'source': {'id': 'S526959', 'version': '1.0.0', 'database': 'mpds'}, 'mc3d_id': 'mc3d-52989', 'source_db': 'mpds', 'source_id': 'S526959', 'duplicates': {'icsd': ['2b6113df-2c28-4e3f-a117-87eee4b99917'], 'mpds': ['bf08ecab-72c3-4523-8393-e2ce1d2673f6']}, 'formula_hill': 'Au6Zr2', 'magnetization': [0.0, 0.0], 'bravais_lattice': 'oP', 'chemical_system': '-Au-Zr-', 'number_of_sites': 8, 'spacegroup_number': 59, 'partial_occupancies': False, 'f

In [10]:
import numpy as np
from aiida.orm import Dict, Group, QueryBuilder, StructureData, WorkChainNode

GROUP_LABEL = 'mc3d-pbesol-v1'

qb = QueryBuilder()
qb.append(Group, filters={'label': GROUP_LABEL}, tag='g')
qb.append(
    WorkChainNode,
    with_group='g',
    filters={
        'attributes.process_label': 'PwBaseWorkChain',
        'attributes.exit_status':   0,
    },
    project=['*'],
    tag='wc',
)
qb.append(
    Dict,
    with_incoming='wc',
    edge_filters={'label': 'output_parameters'},
    filters={'attributes.lsda': True},
    tag='p',
)
qb.append(
    StructureData,
    with_incoming='wc',
    edge_filters={'label': 'output_structure'},
    filters={'extras.chemical_system': {'like': '%-Fe-%'}},
    project=['*'],
    tag='s',
)
qb.order_by({'wc': {'id': 'asc'}})
qb.limit(1)

wc, out_s = qb.first()

# --- 节点 + 结构 ---
print(f'━━━ WC ━━━')
print(f'  pk={wc.pk}  uuid={wc.uuid}  exit={wc.exit_status}  ctime={wc.ctime}')

print(f'\n━━━ OUTPUT structure extras ━━━')
for k, v in sorted(out_s.base.extras.all.items()):
    print(f'  {k:30s} = {v}')

# --- magnetism 相关的 output_parameters ---
p = wc.outputs.output_parameters.get_dict()
mag_keys = [
    'lsda', 'do_magnetization', 'non_colinear_calculation',
    'spin_orbit_calculation', 'spin_orbit_domag',
    'number_of_spin_components',
    'starting_magnetization', 'constraint_mag',
    'magnetization_angle1', 'magnetization_angle2',
    'total_magnetization', 'absolute_magnetization',
    'number_of_electrons', 'number_of_bands',
    'fermi_energy',
]
print(f'\n━━━ output_parameters (magnetism subset) ━━━')
for k in mag_keys:
    print(f'  {k:30s} = {p.get(k, "<missing>")}')

# --- trajectory 里的 atomic_magnetic_moments ---
traj = wc.outputs.output_trajectory
names = traj.get_arraynames()
print(f'\n━━━ output_trajectory arrays ━━━')
for nm in names:
    a = traj.get_array(nm)
    print(f'  {nm:30s} shape={str(a.shape):20s} dtype={a.dtype}')

if 'atomic_magnetic_moments' in names:
    sp = list(traj.get_array('atomic_species_name'))
    m  = traj.get_array('atomic_magnetic_moments')
    print(f'\n━━━ atomic_magnetic_moments (full) ━━━')
    print(f'  shape={m.shape}  (steps, n_atoms)')
    print(f'  species: {sp}')
    print(f'  step 0  (initial)  : {np.round(m[0], 4).tolist()}')
    print(f'  step -1 (converged): {np.round(m[-1], 4).tolist()}')
    converged = m[-1]
    print(f'\n  per-atom converged moments:')
    for i, (s, mv) in enumerate(zip(sp, converged)):
        flag = '  <-- magnetic' if abs(mv) > 0.1 else ''
        print(f'    atom {i:>2}  {s:>3}  {mv:+.4f}{flag}')
    print(f'\n  signed sum    = {converged.sum():+.4f}  (≈ total_magnetization)')
    print(f'  abs sum       = {np.abs(converged).sum():.4f}  (≈ absolute_magnetization)')
    print(f'  max |atomic|  = {np.abs(converged).max():.4f}')
else:
    print('\n  atomic_magnetic_moments: <not in trajectory>')

# --- bands ---
band = wc.outputs.output_band
bands = band.get_array('bands')
print(f'\n━━━ output_band ━━━')
print(f'  bands shape    = {bands.shape}')
print(f'    (k-points, bands)         若 nspin=1')
print(f'    (n_spin, k-points, bands) 若 nspin=2')


━━━ WC ━━━
  pk=1123  uuid=01f4a239-8d48-42e4-b6a2-4ce500954c3e  exit=0  ctime=2021-09-23 18:49:42.654600+01:00

━━━ OUTPUT structure extras ━━━
  bravais_lattice                = tI
  bravais_lattice_extended       = tI2
  chemical_system                = -Ba-Fe-O-W-
  duplicates                     = {'cod': ['8f526053-123a-4f2f-97c1-c9130fb33d48'], 'mpds': ['0db67fa7-96e1-4642-961f-ca0374c537d7']}
  formula_hill                   = Ba2FeO6W
  formula_hill_compact           = Ba2FeO6W
  magnetization                  = [-0.00065664599426977, 0.26040229746084, 0.013839671464471, -0.0089439562040547]
  mc3d_id                        = mc3d-34014
  number_of_sites                = 10
  partial_occupancies            = False
  source                         = {'id': 'S1411816', 'version': '1.0.0', 'database': 'mpds'}
  source_db                      = mpds
  source_id                      = S1411816
  spacegroup_international       = I4/m
  spacegroup_number              = 87

━━━ output

In [6]:
import time
import numpy as np
import pandas as pd
from collections import Counter
from aiida.orm import Dict, Group, QueryBuilder, StructureData, TrajectoryData, WorkChainNode

GROUP_LABEL = 'mc3d-pbesol-v1'

# Thresholds (与 v2 一致)
NM_MAX_ATOMIC   = 0.05
AFM_TOTAL_MAX   = 0.05
SIGNIF_ATOMIC   = 0.10
FM_STRONG_TOTAL = 0.50

# --- Step 1: 数 lsda=False 的（MC3D 跳过 spin 的，构造性 NM）---
qb = QueryBuilder()
qb.append(Group, filters={'label': GROUP_LABEL}, tag='g')
qb.append(WorkChainNode, with_group='g',
          filters={'attributes.process_label': 'PwBaseWorkChain',
                   'attributes.exit_status': 0}, tag='wc')
qb.append(Dict, with_incoming='wc',
          edge_filters={'label': 'output_parameters'},
          filters={'attributes.lsda': False})
n_skip_nm = qb.count()
print(f'NM-by-skip (lsda=False) WCs : {n_skip_nm}')

# --- Step 2: lsda=True WCs 拉 extras + trajectory ---
qb = QueryBuilder()
qb.append(Group, filters={'label': GROUP_LABEL}, tag='g')
qb.append(WorkChainNode, with_group='g',
          filters={'attributes.process_label': 'PwBaseWorkChain',
                   'attributes.exit_status': 0}, tag='wc')
qb.append(Dict, with_incoming='wc',
          edge_filters={'label': 'output_parameters'},
          filters={'attributes.lsda': True}, tag='p')
qb.append(StructureData, with_incoming='wc',
          edge_filters={'label': 'output_structure'},
          project=['extras.mc3d_id', 'extras.formula_hill_compact',
                   'extras.spacegroup_number', 'extras.number_of_sites',
                   'extras.chemical_system'], tag='s')
qb.append(TrajectoryData, with_incoming='wc',
          edge_filters={'label': 'output_trajectory'},
          project=['*'], tag='traj')

n_spin_wc = qb.count()
print(f'spin-polarized (lsda=True) WCs: {n_spin_wc}')
print(f'\nIterating {n_spin_wc} trajectories... (~3-4 min)')

t0 = time.time()
rows = []
for mc3d_id, formula, spg, n_sites, chem, traj in qb.iterall():
    names = traj.get_arraynames()
    if 'atomic_magnetic_moments' not in names:
        continue
    m = traj.get_array('atomic_magnetic_moments')[-1]
    species = (list(traj.get_array('atomic_species_name'))
               if 'atomic_species_name' in names else None)

    # QE 全局值（比 atomic 求和准 — Löwdin 抓不到间隙自旋）
    total_qe = (float(traj.get_array('total_magnetization')[-1])
                if 'total_magnetization' in names else None)
    abs_qe   = (float(traj.get_array('absolute_magnetization')[-1])
                if 'absolute_magnetization' in names else None)

    m_abs = np.abs(m)
    mag_atoms = ([sp for sp, mm in zip(species, m_abs) if mm > SIGNIF_ATOMIC]
                 if species is not None else [])

    rows.append({
        'mc3d_id':         mc3d_id,
        'formula':         formula,
        'spacegroup':      spg,
        'n_sites':         n_sites,
        'chemical_system': chem,
        'species':          species,
        'atomic_moments':   m.tolist(),
        'total_mag_atomic': float(m.sum()),
        'abs_mag_atomic':   float(m_abs.sum()),
        'total_mag_qe':     total_qe,
        'abs_mag_qe':       abs_qe,
        'max_abs_atomic':   float(m_abs.max()),
        'magnetic_species': sorted(set(mag_atoms)),
    })

print(f'\nPulled {len(rows)} rows in {time.time()-t0:.1f}s')

# --- Step 3: 分类 ---
def categorize(r):
    tot_src = r['total_mag_qe'] if r['total_mag_qe'] is not None else r['total_mag_atomic']
    tot = abs(tot_src)
    mx  = r['max_abs_atomic']
    if mx < NM_MAX_ATOMIC and tot < AFM_TOTAL_MAX:
        return 'NM_after_spin_run'        # 跑了 spin 但收敛回 NM
    if tot < AFM_TOTAL_MAX and mx > SIGNIF_ATOMIC:
        return 'AFM_like'                  # local moments 互相抵消
    if tot > FM_STRONG_TOTAL:
        return 'FM_strong'                 # 强 net moment
    if mx > SIGNIF_ATOMIC:
        return 'FiM_or_partial'            # net 非零但不强
    return 'borderline'                    # 微弱

df = pd.DataFrame(rows)
df['category'] = df.apply(categorize, axis=1)

# --- Step 4: 总览 ---
total_wc = len(df) + n_skip_nm
print(f'\n━━━ Category counts (over {total_wc} successful PwBase WCs) ━━━')
print(f'  NM_by_skip          {n_skip_nm:>6}  (lsda=False, MC3D 不开 spin)')
for cat, n in df['category'].value_counts().items():
    print(f'  {cat:20s} {n:>6}')

n_nm = n_skip_nm + (df['category'] == 'NM_after_spin_run').sum()
n_mag = total_wc - n_nm
print(f'\nMagnetic (any) : {n_mag:>6}  ({100*n_mag/total_wc:.1f}%)')
print(f'Nonmagnetic    : {n_nm:>6}  ({100*n_nm/total_wc:.1f}%)')

# --- Step 5: 磁性元素 top 列表 ---
print(f'\n━━━ Top magnetic species (in magnetic structures) ━━━')
mag_df = df[~df['category'].isin(['NM_after_spin_run'])]
species_counter = Counter()
for sp_list in mag_df['magnetic_species']:
    species_counter.update(sp_list)
for sp, c in species_counter.most_common(20):
    print(f'  {sp:>4}  {c:>5}  ({100*c/len(mag_df):.1f}% of magnetic)')

# --- Step 6: 分布 + 样本 ---
print(f'\n━━━ |total_mag_qe| percentiles ━━━')
print(df['total_mag_qe'].abs().describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

print(f'\n━━━ Sample of FM_strong (top 10 by |total_mag_qe|) ━━━')
print(df[df['category'] == 'FM_strong']
      .sort_values('total_mag_qe', key=abs, ascending=False)
      [['mc3d_id', 'formula', 'spacegroup',
        'total_mag_qe', 'abs_mag_qe', 'magnetic_species']]
      .head(10).to_string(index=False))

print(f'\n━━━ Sample of AFM_like (top 10 by max_abs_atomic) ━━━')
print(df[df['category'] == 'AFM_like']
      .sort_values('max_abs_atomic', ascending=False)
      [['mc3d_id', 'formula', 'spacegroup',
        'total_mag_qe', 'max_abs_atomic', 'magnetic_species']]
      .head(10).to_string(index=False))

# --- Step 7: 落盘 ---
df.to_pickle('data/processed/mc3d-pbesol-v1-magnetism.pkl')
print(f'\n✓ wrote data/processed/mc3d-pbesol-v1-magnetism.pkl  ({len(df)} rows)')


NM-by-skip (lsda=False) WCs : 24047
spin-polarized (lsda=True) WCs: 8913

Iterating 8913 trajectories... (~3-4 min)

Pulled 8913 rows in 172.2s

━━━ Category counts (over 32960 successful PwBase WCs) ━━━
  NM_by_skip           24047  (lsda=False, MC3D 不开 spin)
  FM_strong              7193
  NM_after_spin_run      1606
  FiM_or_partial           63
  borderline               41
  AFM_like                 10

Magnetic (any) :   7307  (22.2%)
Nonmagnetic    :  25653  (77.8%)

━━━ Top magnetic species (in magnetic structures) ━━━
    Mn   1349  (18.5% of magnetic)
    Fe   1285  (17.6% of magnetic)
     O    916  (12.5% of magnetic)
    O1    916  (12.5% of magnetic)
    Co    760  (10.4% of magnetic)
    Cr    590  (8.1% of magnetic)
    Ni    477  (6.5% of magnetic)
     V    469  (6.4% of magnetic)
    Cu    430  (5.9% of magnetic)
     N    323  (4.4% of magnetic)
    O0    276  (3.8% of magnetic)
     F    213  (2.9% of magnetic)
    Ti    146  (2.0% of magnetic)
    Cl    136  (1.9%

In [7]:
import re
from collections import Counter
from aiida.orm import Group, QueryBuilder, StructureData, WorkChainNode

# Issue 1: 把 Fe0/Fe1/O0/O1 这种带数字后缀的 kind 名归并成元素符号
def strip_kind(name):
    return re.sub(r'\d+$', '', str(name))

mag_df = df[~df['category'].isin(['NM_after_spin_run'])]
clean_counter = Counter()
for sp_list in mag_df['magnetic_species']:
    clean_counter.update(strip_kind(s) for s in sp_list)

print(f'━━━ Top magnetic ELEMENTS (stripped, n_magnetic={len(mag_df)}) ━━━')
for sp, c in clean_counter.most_common(15):
    print(f'  {sp:>4}  {c:>5}  ({100*c/len(mag_df):5.1f}% of magnetic)')

# Issue 2: 哪些 output_structure 没有 mc3d_id？
qb = QueryBuilder()
qb.append(Group, filters={'label': 'mc3d-pbesol-v1'}, tag='g')
qb.append(WorkChainNode, with_group='g',
          filters={'attributes.process_label': 'PwBaseWorkChain',
                   'attributes.exit_status': 0},
          project=['*'], tag='wc')
qb.append(StructureData, with_incoming='wc',
          edge_filters={'label': 'output_structure'},
          filters={'extras': {'!has_key': 'mc3d_id'}},
          project=['*'], tag='s')

print(f'\n━━━ output_structures 缺 mc3d_id 的 WCs ━━━')
missing_pks = []
for wc, s in qb.iterall():
    missing_pks.append(wc.pk)
    print(f'  WC pk={wc.pk}  out_s pk={s.pk}  formula={s.get_formula()}')
    print(f'    extras keys: {sorted(s.base.extras.all.keys())}')
print(f'\n  共 {len(missing_pks)} 个')


━━━ Top magnetic ELEMENTS (stripped, n_magnetic=7307) ━━━
     O   2235  ( 30.6% of magnetic)
    Mn   1530  ( 20.9% of magnetic)
    Fe   1493  ( 20.4% of magnetic)
    Co    905  ( 12.4% of magnetic)
    Cr    641  (  8.8% of magnetic)
     V    527  (  7.2% of magnetic)
    Ni    495  (  6.8% of magnetic)
    Cu    478  (  6.5% of magnetic)
     N    374  (  5.1% of magnetic)
     F    283  (  3.9% of magnetic)
    Ti    171  (  2.3% of magnetic)
    Cl    160  (  2.2% of magnetic)
    Ru    133  (  1.8% of magnetic)
    Mo    122  (  1.7% of magnetic)
    Ir     78  (  1.1% of magnetic)

━━━ output_structures 缺 mc3d_id 的 WCs ━━━
  WC pk=910  out_s pk=14491  formula=Bi2O6Sr
    extras keys: []
  WC pk=1194  out_s pk=15444  formula=Ag2Cu2Se2
    extras keys: []
  WC pk=1346  out_s pk=37825  formula=Ag2Ba2Bi2
    extras keys: []
  WC pk=2902  out_s pk=15227  formula=AgCl6Cs2Sb
    extras keys: []
  WC pk=3763  out_s pk=45192  formula=AlO4P
    extras keys: []
  WC pk=5710  out_s pk=48

In [8]:
import pandas as pd
from collections import Counter
from aiida.orm import load_node, Group, QueryBuilder, StructureData, WorkChainNode

# 先看看 exit=501 在 PwBase 里是什么意思
from aiida_quantumespresso.workflows.pw.base import PwBaseWorkChain
spec_codes = PwBaseWorkChain.spec().exit_codes
print('PwBaseWorkChain 5xx exit codes:')
for label in dir(spec_codes):
    if label.startswith('_'):
        continue
    code = getattr(spec_codes, label, None)
    if code is None or not hasattr(code, 'status'):
        continue
    if 500 <= code.status <= 510:
        print(f'  {code.status}  {label}: {code.message}')

# 拉所有 501 失败的 + 它们的 input structure（注意：可能没 output_structure）
qb = QueryBuilder()
qb.append(Group, filters={'label': 'mc3d-pbesol-v1'}, tag='g')
qb.append(WorkChainNode, with_group='g',
          filters={'attributes.process_label': 'PwBaseWorkChain',
                   'attributes.exit_status': 501},
          project=['*'], tag='wc')
qb.append(StructureData, with_outgoing='wc',
          edge_filters={'label': 'pw__structure'},
          project=['extras.mc3d_id', 'extras.formula_hill_compact',
                   'extras.spacegroup_number', 'extras.number_of_sites',
                   'extras.chemical_system'], tag='s')

fail_rows = []
for wc, mc3d_id, formula, spg, n_sites, chem in qb.iterall():
    fail_rows.append({
        'pk': wc.pk, 'mc3d_id': mc3d_id, 'formula': formula,
        'spacegroup': spg, 'n_sites': n_sites, 'chemical_system': chem,
    })
fail_df = pd.DataFrame(fail_rows)
print(f'\nFailed WCs collected: {len(fail_df)}')
print(f'\nn_sites distribution:')
print(fail_df['n_sites'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())

# 元素富集分析：哪些元素在失败里超额出现
fail_elem = Counter()
for chem in fail_df['chemical_system'].dropna():
    fail_elem.update(e for e in chem.split('-') if e)

# 全 v1 元素频率作 baseline
qb = QueryBuilder()
qb.append(Group, filters={'label': 'mc3d-pbesol-v1'}, tag='g')
qb.append(WorkChainNode, with_group='g',
          filters={'attributes.process_label': 'PwBaseWorkChain'}, tag='wc')
qb.append(StructureData, with_incoming='wc',
          edge_filters={'label': 'output_structure'},
          project=['extras.chemical_system'])
overall_elem = Counter()
n_overall = 0
for (chem,) in qb.iterall():
    if chem:
        overall_elem.update(e for e in chem.split('-') if e)
        n_overall += 1

print(f'\n━━━ Element enrichment in failures (sorted by ratio, baseline n>=100) ━━━')
print(f'  {"elem":<6} {"n_fail":>7} {"%fail":>7} {"%overall":>9} {"ratio":>7}')
ratios = []
for elem, nf in fail_elem.items():
    no = overall_elem.get(elem, 0)
    if no < 100:
        continue
    ratios.append((elem, nf, nf/len(fail_df), no/n_overall, (nf/len(fail_df))/(no/n_overall)))
ratios.sort(key=lambda x: x[4], reverse=True)
for elem, nf, ff, fo, r in ratios[:15]:
    print(f'  {elem:<6} {nf:>7} {100*ff:>6.1f}% {100*fo:>8.1f}% {r:>6.2f}x')

# 看一个失败的 WC 长啥样
print(f'\n━━━ Inspecting first failed WC ━━━')
wc = load_node(fail_df.iloc[0]['pk'])
print(f'  pk={wc.pk}  uuid={wc.uuid}')
print(f'  exit_status={wc.exit_status}  exit_message={wc.exit_message}')
print(f'  inputs.pw.structure formula={wc.inputs.pw.structure.get_formula()}')
print(f'  outputs available: {list(wc.outputs)}')


PwBaseWorkChain 5xx exit codes:
  501  ERROR_IONIC_CONVERGENCE_REACHED_EXCEPT_IN_FINAL_SCF: Then ionic minimization cycle converged but the thresholds are exceeded in the final SCF.

Failed WCs collected: 2084

n_sites distribution:
count    257.000000
mean      18.891051
std       15.429141
min        1.000000
50%       14.000000
75%       24.000000
90%       44.000000
95%       49.200000
99%       64.880000
max       70.000000

━━━ Element enrichment in failures (sorted by ratio, baseline n>=100) ━━━
  elem    n_fail   %fail  %overall   ratio
  C           66    3.2%      9.3%   0.34x
  W           11    0.5%      2.0%   0.27x
  Fe          22    1.1%      5.6%   0.19x
  Be           4    0.2%      1.2%   0.16x
  I           10    0.5%      3.1%   0.15x
  N           36    1.7%     11.4%   0.15x
  Cl          22    1.1%      7.2%   0.15x
  Ag           9    0.4%      3.0%   0.14x
  F           26    1.2%      8.7%   0.14x
  Ni          15    0.7%      5.0%   0.14x
  Pd           8   

In [9]:
# v1 lsda=True 的 8,913 个 — 和 v2 全 33,142 个直接比
print(f'━━━ v1 (spin-polarized only, n={len(df)}) ━━━')
for cat, n in df['category'].value_counts().items():
    print(f'  {cat:20s} {n:>6}  ({100*n/len(df):5.1f}%)')

print(f'\n━━━ v2 (all, n=33,142, also all lsda=True) ━━━')
v2_counts = {'NM': 27435, 'FM_strong': 5521, 'FiM_or_partial': 159,
             'borderline': 27, 'AFM_like': 0}
v2_total = sum(v2_counts.values())
for cat, n in v2_counts.items():
    print(f'  {cat:20s} {n:>6}  ({100*n/v2_total:5.1f}%)')

# 在同方法论（都跑了 spin）下的 magnetic 比例
v1_mag_frac = (df['category'] != 'NM_after_spin_run').mean()
v2_mag_frac = 1 - v2_counts['NM']/v2_total
print(f'\n━━━ Magnetic fraction (apples-to-apples) ━━━')
print(f'  v1 (8,913 spin runs): {100*v1_mag_frac:.1f}%')
print(f'  v2 (33,142 spin runs): {100*v2_mag_frac:.1f}%')
print(f'\n→ v1 的 80%+ 命中率说明 MC3D 的 pre-filter 很准 (省了 73% 计算)')
print(f'→ v2 全跑 spin 是为了零漏 — 代价是 5x 计算量、命中率 17%')


━━━ v1 (spin-polarized only, n=8913) ━━━
  FM_strong              7193  ( 80.7%)
  NM_after_spin_run      1606  ( 18.0%)
  FiM_or_partial           63  (  0.7%)
  borderline               41  (  0.5%)
  AFM_like                 10  (  0.1%)

━━━ v2 (all, n=33,142, also all lsda=True) ━━━
  NM                    27435  ( 82.8%)
  FM_strong              5521  ( 16.7%)
  FiM_or_partial          159  (  0.5%)
  borderline               27  (  0.1%)
  AFM_like                  0  (  0.0%)

━━━ Magnetic fraction (apples-to-apples) ━━━
  v1 (8,913 spin runs): 82.0%
  v2 (33,142 spin runs): 17.2%

→ v1 的 80%+ 命中率说明 MC3D 的 pre-filter 很准 (省了 73% 计算)
→ v2 全跑 spin 是为了零漏 — 代价是 5x 计算量、命中率 17%


In [13]:
import pandas as pd
from aiida.orm import Group, QueryBuilder, StructureData, WorkChainNode

qb = QueryBuilder()
qb.append(Group, filters={'label': 'mc3d-pbesol-v1'}, tag='g')
qb.append(WorkChainNode, with_group='g',
          filters={'attributes.process_label': 'PwBaseWorkChain',
                   'attributes.exit_status': 0}, tag='wc')
qb.append(StructureData, with_incoming='wc',
          edge_filters={'label': 'output_structure'},
          filters={'extras': {'has_key': 'mc3d_id'}},
          project=['extras.mc3d_id',
                   'extras.source_db', 'extras.source_id',
                   'extras.formula_hill',
                   'extras.bravais_lattice',
                   'extras.spacegroup_international',
                   'extras.partial_occupancies'])

extras_rows = []
for mc3d_id, sdb, sid, fh, bl, spg_i, po in qb.iterall():
    extras_rows.append({
        'mc3d_id':                  mc3d_id,
        'source_db':                sdb,
        'source_db_id':             sid,
        'formula_hill':             fh,
        'bravais_lattice':          bl,
        'spacegroup_international': spg_i,
        'partial_occupancies':      po,
    })
extras_df = pd.DataFrame(extras_rows).drop_duplicates(subset=['mc3d_id'])
print(f'拉到 {len(extras_df)} 个 structure extras')

# 关键：合并前剔掉 df 里可能存在的旧列（避免 _x/_y 冲突）
to_drop = [c for c in extras_df.columns if c != 'mc3d_id' and c in df.columns]
# 也清掉之前可能产生的 _x/_y 残留
to_drop = [c for c in extras_df.columns if c != 'mc3d_id' and c in df.columns]
to_drop += [c for c in df.columns if c.endswith('_x') or c.endswith('_y')]
to_drop += [c for c in ('source_id',) if c in df.columns]   # ← 显式清旧名
df_clean = df.drop(columns=list(set(to_drop)))

if to_drop:
    print(f'剔掉 df 里的旧列: {sorted(set(to_drop))}')

df_enriched = df_clean.merge(extras_df, on='mc3d_id', how='left')
print(f'df: {len(df)} -> df_enriched: {len(df_enriched)}')
print(f'最终列: {sorted(df_enriched.columns)}')

print(f'\nsource_db 分布:')
print(df_enriched['source_db'].value_counts(dropna=False).to_string())

print(f'\nsource_db × category 交叉表:')
print(pd.crosstab(df_enriched['source_db'].fillna('<missing>'),
                  df_enriched['category']).to_string())

df = df_enriched
df.to_pickle('data/processed/mc3d-pbesol-v1-magnetism.pkl')
print(f'\n✓ overwrote pkl ({len(df)} rows × {df.shape[1]} cols)')


拉到 32430 个 structure extras
剔掉 df 里的旧列: ['bravais_lattice', 'formula_hill', 'partial_occupancies', 'source_db', 'source_db_id', 'source_id', 'spacegroup_international']
df: 8913 -> df_enriched: 8913
最终列: ['abs_mag_atomic', 'abs_mag_qe', 'atomic_moments', 'bravais_lattice', 'category', 'chemical_system', 'formula', 'formula_hill', 'magnetic_species', 'max_abs_atomic', 'mc3d_id', 'n_sites', 'partial_occupancies', 'source_db', 'source_db_id', 'spacegroup', 'spacegroup_international', 'species', 'total_mag_atomic', 'total_mag_qe']

source_db 分布:
source_db
mpds    6937
icsd    1606
cod      203
NaN      167

source_db × category 交叉表:
category   AFM_like  FM_strong  FiM_or_partial  NM_after_spin_run  borderline
source_db                                                                    
<missing>         1        100               2                 63           1
cod               1        127               2                 70           3
icsd              5       1196              20     

---
## D — v1 vs v2 magnetism comparison

In [13]:
from aiida import load_profile
load_profile()

import pandas as pd

PKL_DIR = 'data/processed'
df_v1 = pd.read_pickle(f'{PKL_DIR}/mc3d-pbesol-v1-magnetism.pkl')
df_v2 = pd.read_pickle(f'{PKL_DIR}/mc3d-pbesol-v2-magnetism.pkl')

print(f'v1: rows={len(df_v1)}  cols={list(df_v1.columns)}')
print(f'v2: rows={len(df_v2)}  cols={list(df_v2.columns)}')
print(f'\nv1 head:\n{df_v1.head(2)}')
print(f'\nv2 head:\n{df_v2.head(2)}')


v1: rows=8913  cols=['mc3d_id', 'formula', 'spacegroup', 'n_sites', 'chemical_system', 'species', 'atomic_moments', 'total_mag_atomic', 'abs_mag_atomic', 'total_mag_qe', 'abs_mag_qe', 'max_abs_atomic', 'magnetic_species', 'category', 'source_db', 'source_db_id', 'formula_hill', 'bravais_lattice', 'spacegroup_international', 'partial_occupancies']
v2: rows=33142  cols=['mc3d_id', 'formula', 'spacegroup', 'n_sites', 'chemical_system', 'source_db', 'species', 'atomic_moments', 'total_mag', 'abs_mag', 'max_abs_atomic', 'magnetic_species', 'category', 'source_db_id']

v1 head:
      mc3d_id formula  spacegroup  n_sites chemical_system  \
0    mc3d-851  AsMnRh       189.0      9.0      -As-Mn-Rh-   
1  mc3d-59537    AsCr       194.0      4.0         -As-Cr-   

                                species  \
0  [Mn, Mn, Mn, As, As, As, Rh, Rh, Rh]   
1                      [Cr, Cr, As, As]   

                                      atomic_moments  total_mag_atomic  \
0  [3.2581, 3.2581, 3.2581, -0

In [15]:
import pandas as pd
import numpy as np

base = 'data/processed'
df_v1 = pd.read_pickle(f'{base}/mc3d-pbesol-v1-magnetism.pkl')
df_v2 = pd.read_pickle(f'{base}/mc3d-pbesol-v2-magnetism.pkl')

# Normalize v2: strip the redundant '{source_db}-' prefix so source_db_id
# means the same thing in both pkls (= within-DB id).
def strip_prefix(row):
    sid = row['source_db_id']
    pref = f"{row['source_db']}-"
    return sid[len(pref):] if isinstance(sid, str) and sid.startswith(pref) else sid

df_v2['source_db_id'] = df_v2.apply(strip_prefix, axis=1)
print('After normalization, v2 source_db_id sample:')
print(df_v2[['mc3d_id', 'source_db', 'source_db_id']].head(3).to_string())

# Magnetic-flag (v1 uses NM_after_spin_run, v2 uses NM)
NM_LABELS = {'NM', 'NM_after_spin_run'}
df_v1['is_magnetic'] = ~df_v1['category'].isin(NM_LABELS)
df_v2['is_magnetic'] = ~df_v2['category'].isin(NM_LABELS)

mag_v1 = df_v1[df_v1['is_magnetic']].copy()
mag_v2 = df_v2[df_v2['is_magnetic']].copy()
print(f'\nmagnetic counts:  v1={len(mag_v1)}  v2={len(mag_v2)}')

# Join key = (source_db, source_db_id)
KEY = ['source_db', 'source_db_id']

# Set membership on the key
keys_v1 = set(map(tuple, mag_v1[KEY].itertuples(index=False, name=None)))
keys_v2 = set(map(tuple, mag_v2[KEY].itertuples(index=False, name=None)))
both = keys_v1 & keys_v2
only_v1 = keys_v1 - keys_v2
only_v2 = keys_v2 - keys_v1
print(f'\nmagnetic in both         : {len(both)}')
print(f'magnetic only in v1      : {len(only_v1)}')
print(f'magnetic only in v2      : {len(only_v2)}')

# Per-source-DB breakdown
def split_by_db(s):
    from collections import Counter
    return dict(Counter(db for db, _ in s))
print(f'\nonly_v1 by source_db: {split_by_db(only_v1)}')
print(f'only_v2 by source_db: {split_by_db(only_v2)}')
print(f'both     by source_db: {split_by_db(both)}')

# Inner join — overlap rows side-by-side for direct comparison
joined = pd.merge(
    mag_v1[KEY + ['mc3d_id', 'formula', 'category', 'total_mag_atomic',
                  'max_abs_atomic', 'magnetic_species']]
        .rename(columns={'mc3d_id': 'mc3d_id_v1', 'category': 'category_v1',
                         'total_mag_atomic': 'total_mag_v1',
                         'max_abs_atomic': 'max_abs_v1',
                         'magnetic_species': 'mag_species_v1'}),
    mag_v2[KEY + ['mc3d_id', 'category', 'total_mag', 'max_abs_atomic',
                  'magnetic_species']]
        .rename(columns={'mc3d_id': 'mc3d_id_v2', 'category': 'category_v2',
                         'total_mag': 'total_mag_v2',
                         'max_abs_atomic': 'max_abs_v2',
                         'magnetic_species': 'mag_species_v2'}),
    on=KEY, how='inner',
)
print(f'\noverlap join rows: {len(joined)}')

# Sanity: do the two versions agree on which crystal it is? Compare formula.
# (We've stored only formula from v1; v2's formula was dropped from this table.)
# Bring v2 formula back for cross-check:
joined = joined.merge(mag_v2[KEY + ['formula']].rename(columns={'formula': 'formula_v2'}),
                      on=KEY, how='left')
joined['formula_match'] = joined['formula'] == joined['formula_v2']
print(f'formula agreement on overlap: {joined["formula_match"].sum()} / {len(joined)}')

# mc3d_id stability: do (source_db, source_db_id) maps to the SAME mc3d_id in both?
joined['mc3d_id_match'] = joined['mc3d_id_v1'] == joined['mc3d_id_v2']
print(f'mc3d_id stable across versions: {joined["mc3d_id_match"].sum()} / {len(joined)}')

# Compare magnetism magnitudes
joined['delta_total'] = (joined['total_mag_v1'].abs() - joined['total_mag_v2'].abs())
joined['delta_max']   = (joined['max_abs_v1']  - joined['max_abs_v2'])


After normalization, v2 source_db_id sample:
      mc3d_id source_db source_db_id
0  mc3d-19919      icsd       183155
1  mc3d-48310      icsd        49583
2  mc3d-38841      icsd       184928

magnetic counts:  v1=7307  v2=5707

magnetic in both         : 1253
magnetic only in v1      : 6054
magnetic only in v2      : 4454

only_v1 by source_db: {'mpds': 5127, 'icsd': 739, nan: 1, 'cod': 84, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan: 1, nan:

In [16]:
# Diagnose NaN source_db in v1
nan_v1 = df_v1[df_v1['source_db'].isna()]
print(f"v1 rows with NaN source_db: {len(nan_v1)}")
print(nan_v1[['mc3d_id', 'formula', 'source_db', 'source_db_id', 'category']].head(10).to_string())


v1 rows with NaN source_db: 167
     mc3d_id formula source_db source_db_id           category
47       NaN     NaN       NaN          NaN          FM_strong
92       NaN     NaN       NaN          NaN  NM_after_spin_run
164      NaN     NaN       NaN          NaN          FM_strong
174      NaN     NaN       NaN          NaN          FM_strong
267      NaN     NaN       NaN          NaN  NM_after_spin_run
707      NaN     NaN       NaN          NaN  NM_after_spin_run
722      NaN     NaN       NaN          NaN          FM_strong
738      NaN     NaN       NaN          NaN          FM_strong
814      NaN     NaN       NaN          NaN          FM_strong
1015     NaN     NaN       NaN          NaN          FM_strong


In [17]:
# Decompose only_v1: how many are entirely missing from v2 archive vs flipped to NM?
keys_v1_all = set(map(tuple, df_v1[KEY].dropna().itertuples(index=False, name=None)))
keys_v2_all = set(map(tuple, df_v2[KEY].dropna().itertuples(index=False, name=None)))

# subset: rows in v1 magnetic
mag_v1_keys = set(map(tuple, mag_v1[KEY].dropna().itertuples(index=False, name=None)))

flipped_to_nm    = mag_v1_keys & keys_v2_all - keys_v2  # in both archives, v1 mag, v2 NM
dropped_from_v2  = mag_v1_keys - keys_v2_all            # not in v2 archive at all

print(f'v1-magnetic decomposition (total {len(mag_v1_keys)}):')
print(f'  also magnetic in v2     : {len(mag_v1_keys & keys_v2):>5}')
print(f'  in v2 archive but NM    : {len(flipped_to_nm):>5}  ← v2 false negatives (likely starting_mag=0 issue)')
print(f'  not in v2 archive at all: {len(dropped_from_v2):>5}  ← v2 dropped these crystals')

# Same for only_v2
flipped_from_nm = set(map(tuple, mag_v2[KEY].dropna().itertuples(index=False, name=None))) & keys_v1_all - keys_v1
added_in_v2     = set(map(tuple, mag_v2[KEY].dropna().itertuples(index=False, name=None))) - keys_v1_all
print(f'\nv2-magnetic decomposition (total {len(mag_v2)}):')
print(f'  also magnetic in v1     : {len(set(map(tuple, mag_v2[KEY].dropna().itertuples(index=False, name=None))) & keys_v1):>5}')
print(f'  in v1 archive but NM    : {len(flipped_from_nm):>5}  ← v1 didn t mark magnetic, v2 did')
print(f'  not in v1 archive at all: {len(added_in_v2):>5}  ← v2 added these crystals')


v1-magnetic decomposition (total 7203):
  also magnetic in v2     :  1253
  in v2 archive but NM    :   481  ← v2 false negatives (likely starting_mag=0 issue)
  not in v2 archive at all:  5469  ← v2 dropped these crystals

v2-magnetic decomposition (total 5707):
  also magnetic in v1     :  1253
  in v1 archive but NM    :    25  ← v1 didn t mark magnetic, v2 did
  not in v1 archive at all:  4428  ← v2 added these crystals
